# Confidence Intervals and Communicating Uncertainty

The previous notebook gave us a yes-or-no answer: the drug's effect is statistically significant. But as the explainer pointed out, a regulator does not just need to know that the drug works. They need to know whether the true reduction could be as low as 2 points (barely noticeable) or as high as 14 (substantial). The difference between "somewhere between 2 and 14" and "somewhere between 7 and 9" is enormous for a decision, even though both ranges have the same centre.

**Confidence intervals** provide that range. They move us from "the effect exists" to "here is how large the effect plausibly is." In this notebook we will calculate confidence intervals for means and proportions, build bootstrap intervals from scratch, and practise writing results that a non-technical audience can understand.

By the end of this notebook we will be able to:

- Calculate a **confidence interval for a mean** using the t-distribution
- Calculate a **confidence interval for a proportion**
- Build **bootstrap confidence intervals** by resampling with replacement
- Visualise CIs with error bar plots
- Write a plain-language summary of statistical findings

In [ ]:
%pip install -q pandas matplotlib seaborn scipy

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

sns.set_style("whitegrid")
np.random.seed(42)

In [ ]:
patients = pd.read_csv("../data/trial_patients.csv")
outcomes = pd.read_csv("../data/treatment_outcomes.csv")

df = patients.merge(outcomes, on="patient_id")
df["score_change"] = df["week_12_score"] - df["baseline_score"]

treatment = df[df["group"] == "treatment"]
control = df[df["group"] == "control"]

print(f"Treatment: {len(treatment)} patients")
print(f"Control:   {len(control)} patients")

---

## 1. Confidence interval for a mean

The explainer was careful about the correct interpretation: a **95% confidence interval** is constructed so that if we repeated the entire study many times, about 95% of the resulting intervals would contain the true population value. The confidence attaches to the *method*, not to any single interval. Keep that in mind as we work through the calculations.

The formula using the t-distribution is:

$$\bar{x} \pm t_{\alpha/2, \, n-1} \times \frac{s}{\sqrt{n}}$$

where $\bar{x}$ is the sample mean, $s$ is the sample standard deviation, $n$ is the sample size, and $t_{\alpha/2, \, n-1}$ is the critical value from the t-distribution. We will recognise $s / \sqrt{n}$ as the standard error from the first notebook.

In [ ]:
# CI for the treatment group's mean week 12 score
data = treatment["week_12_score"].values
n = len(data)
mean = data.mean()
se = data.std(ddof=1) / np.sqrt(n)

ci_low, ci_high = stats.t.interval(confidence=0.95, df=n - 1, loc=mean, scale=se)

print(f"Treatment week 12 mean: {mean:.2f}")
print(f"Standard error:         {se:.3f}")
print(f"95% CI:                 ({ci_low:.2f}, {ci_high:.2f})")

### Manual calculation

We can also compute the CI step by step. Seeing each component separately makes the mechanics clearer and connects back to the standard error formula we verified in Notebook 1.

In [ ]:
# Step by step
t_crit = stats.t.ppf(0.975, df=n - 1)  # 0.975 because two-tailed
margin_of_error = t_crit * se

ci_manual = (mean - margin_of_error, mean + margin_of_error)

print(f"t critical value (0.975, df={n-1}): {t_crit:.4f}")
print(f"Margin of error: {margin_of_error:.3f}")
print(f"95% CI (manual): ({ci_manual[0]:.2f}, {ci_manual[1]:.2f})")

---

## 2. Comparing CIs between groups

A single interval is useful. Two intervals side by side are more so, because they let us visually assess whether the groups differ. If the intervals do not overlap, the difference is likely real. If they overlap substantially, the evidence is weaker.

In [ ]:
def mean_ci(data, confidence=0.95):
    """Return (mean, ci_low, ci_high) for an array."""
    n = len(data)
    mean = data.mean()
    se = data.std(ddof=1) / np.sqrt(n)
    ci_low, ci_high = stats.t.interval(confidence=confidence, df=n - 1, loc=mean, scale=se)
    return mean, ci_low, ci_high


t_mean, t_lo, t_hi = mean_ci(treatment["week_12_score"].values)
c_mean, c_lo, c_hi = mean_ci(control["week_12_score"].values)

print(f"Treatment: {t_mean:.2f}  95% CI ({t_lo:.2f}, {t_hi:.2f})")
print(f"Control:   {c_mean:.2f}  95% CI ({c_lo:.2f}, {c_hi:.2f})")

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))

groups = ["Treatment", "Control"]
means = [t_mean, c_mean]
ci_lower = [t_mean - t_lo, c_mean - c_lo]
ci_upper = [t_hi - t_mean, c_hi - c_mean]

ax.barh(groups, means, xerr=[ci_lower, ci_upper], capsize=5,
        color=["steelblue", "coral"], edgecolor="white", height=0.5)
ax.set_xlabel("Mean week 12 score")
ax.set_title("Week 12 scores with 95% confidence intervals")
plt.tight_layout()
plt.show()

---

## 3. CI for the difference in means

Comparing two separate intervals gives a rough visual sense, but the proper approach is to build a single CI for the difference between treatment and control. This directly answers the question the regulator cares about: how large is the drug's effect? If this interval excludes zero, the difference is statistically significant at that confidence level.

In [ ]:
t_data = treatment["week_12_score"].values
c_data = control["week_12_score"].values

diff_mean = t_data.mean() - c_data.mean()
se_diff = np.sqrt(t_data.var(ddof=1) / len(t_data) + c_data.var(ddof=1) / len(c_data))

# Welch's approximate degrees of freedom
df_welch = (t_data.var(ddof=1) / len(t_data) + c_data.var(ddof=1) / len(c_data))**2 / (
    (t_data.var(ddof=1) / len(t_data))**2 / (len(t_data) - 1)
    + (c_data.var(ddof=1) / len(c_data))**2 / (len(c_data) - 1)
)

t_crit = stats.t.ppf(0.975, df=df_welch)
ci_diff = (diff_mean - t_crit * se_diff, diff_mean + t_crit * se_diff)

print(f"Difference in means: {diff_mean:.2f}")
print(f"SE of difference:    {se_diff:.3f}")
print(f"95% CI for diff:     ({ci_diff[0]:.2f}, {ci_diff[1]:.2f})")
print()
if ci_diff[0] > 0 or ci_diff[1] < 0:
    print("The CI excludes zero: the difference is statistically significant.")
else:
    print("The CI includes zero: the difference is not statistically significant.")

---

## 4. Confidence interval for a proportion

Not every outcome is a continuous score. The trial also records whether each patient improved, stayed stable, or worsened. What proportion of treatment patients improved, and how precise is that estimate?

For proportions, use the normal approximation:

$$\hat{p} \pm z_{\alpha/2} \times \sqrt{\frac{\hat{p}(1 - \hat{p})}{n}}$$

In [ ]:
n_treatment = len(treatment)
n_improved = (treatment["final_outcome"] == "improved").sum()
p_hat = n_improved / n_treatment

z_crit = stats.norm.ppf(0.975)
se_prop = np.sqrt(p_hat * (1 - p_hat) / n_treatment)
ci_prop = (p_hat - z_crit * se_prop, p_hat + z_crit * se_prop)

print(f"Improved (treatment): {n_improved}/{n_treatment} = {p_hat:.3f}")
print(f"95% CI for proportion: ({ci_prop[0]:.3f}, {ci_prop[1]:.3f})")

In [ ]:
# Compare improvement proportions for both groups
n_control = len(control)
n_improved_c = (control["final_outcome"] == "improved").sum()
p_hat_c = n_improved_c / n_control
se_prop_c = np.sqrt(p_hat_c * (1 - p_hat_c) / n_control)
ci_prop_c = (p_hat_c - z_crit * se_prop_c, p_hat_c + z_crit * se_prop_c)

print(f"Treatment improvement rate: {p_hat:.3f}  95% CI ({ci_prop[0]:.3f}, {ci_prop[1]:.3f})")
print(f"Control improvement rate:   {p_hat_c:.3f}  95% CI ({ci_prop_c[0]:.3f}, {ci_prop_c[1]:.3f})")

---

## 5. Bootstrap confidence intervals

The methods above rely on distributional assumptions (normality, large samples). **Bootstrapping** is a non-parametric alternative that works by resampling the data we already have. The idea is beautifully simple: if our sample is the best picture we have of the population, then drawing new samples *from our sample* (with replacement) simulates what would happen if we could repeat the study.

The procedure:

1. Draw a sample of size $n$ **with replacement** from the data.
2. Compute the statistic of interest (e.g. the mean).
3. Repeat thousands of times.
4. Take the 2.5th and 97.5th percentiles for a 95% CI.

In [ ]:
def bootstrap_ci(data, stat_func=np.mean, n_boot=10000, confidence=0.95):
    """Compute a bootstrap confidence interval for a given statistic."""
    boot_stats = []
    n = len(data)
    for _ in range(n_boot):
        resample = np.random.choice(data, size=n, replace=True)
        boot_stats.append(stat_func(resample))
    boot_stats = np.array(boot_stats)
    alpha = 1 - confidence
    lower = np.percentile(boot_stats, 100 * alpha / 2)
    upper = np.percentile(boot_stats, 100 * (1 - alpha / 2))
    return boot_stats, lower, upper

In [ ]:
# Bootstrap CI for the treatment group mean
boot_dist, boot_lo, boot_hi = bootstrap_ci(treatment["week_12_score"].values)

print(f"Bootstrap 95% CI for treatment mean: ({boot_lo:.2f}, {boot_hi:.2f})")
print(f"Analytical 95% CI:                   ({t_lo:.2f}, {t_hi:.2f})")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(boot_dist, bins=60, edgecolor="white", alpha=0.7, density=True)
ax.axvline(boot_lo, color="red", linestyle="--", linewidth=2, label=f"2.5th percentile = {boot_lo:.2f}")
ax.axvline(boot_hi, color="red", linestyle="--", linewidth=2, label=f"97.5th percentile = {boot_hi:.2f}")
ax.axvline(treatment["week_12_score"].mean(), color="black", linewidth=2, label="Sample mean")
ax.set_xlabel("Bootstrap mean")
ax.set_ylabel("Density")
ax.set_title("Bootstrap distribution of the treatment group mean")
ax.legend()
plt.tight_layout()
plt.show()

### Bootstrap CI for the difference in means

Bootstrap is particularly useful for statistics that do not have neat analytical formulas. The difference in means does have one (we used it in section 3), but the bootstrap approach generalises to medians, ratios, and other quantities where closed-form solutions are harder to come by.

In [ ]:
t_vals = treatment["week_12_score"].values
c_vals = control["week_12_score"].values

boot_diffs = []
for _ in range(10000):
    t_resample = np.random.choice(t_vals, size=len(t_vals), replace=True)
    c_resample = np.random.choice(c_vals, size=len(c_vals), replace=True)
    boot_diffs.append(t_resample.mean() - c_resample.mean())

boot_diffs = np.array(boot_diffs)
diff_lo = np.percentile(boot_diffs, 2.5)
diff_hi = np.percentile(boot_diffs, 97.5)

print(f"Bootstrap 95% CI for difference: ({diff_lo:.2f}, {diff_hi:.2f})")
print(f"Analytical 95% CI:               ({ci_diff[0]:.2f}, {ci_diff[1]:.2f})")

---

## 6. Visualising CIs across trial sites

The trial ran across multiple sites, and each site enrolled its own set of patients. Error bar plots are an effective way to show CIs for multiple subgroups at once, making it easy to spot whether the treatment effect is consistent or whether some sites tell a different story.

In [ ]:
site_results = []
for site_id in sorted(df["site_id"].unique()):
    for group_name in ["treatment", "control"]:
        subset = df[(df["site_id"] == site_id) & (df["group"] == group_name)]
        m, lo, hi = mean_ci(subset["week_12_score"].values)
        site_results.append({
            "site_id": site_id,
            "group": group_name,
            "mean": m,
            "ci_low": lo,
            "ci_high": hi,
        })

site_df = pd.DataFrame(site_results)
site_df.head()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

offset = 0.15
for i, group_name in enumerate(["treatment", "control"]):
    subset = site_df[site_df["group"] == group_name]
    positions = subset["site_id"].values + (i - 0.5) * offset
    errors = [subset["mean"] - subset["ci_low"], subset["ci_high"] - subset["mean"]]
    colour = "steelblue" if group_name == "treatment" else "coral"
    ax.errorbar(positions, subset["mean"], yerr=errors, fmt="o", capsize=4,
                color=colour, label=group_name.capitalize(), markersize=6)

ax.set_xlabel("Site ID")
ax.set_ylabel("Mean week 12 score")
ax.set_title("Week 12 scores by site with 95% CIs")
ax.set_xticks(sorted(df["site_id"].unique()))
ax.legend()
plt.tight_layout()
plt.show()

---

## 7. Writing results for a non-technical audience

The explainer made the case that communicating uncertainty is a professional skill. Research shows that presenting numbers with uncertainty ranges does not reduce public trust; what reduces trust is when a "definitive" number later turns out to be wrong. So the honest thing and the strategically wise thing are the same: lead with the estimate, follow with the range, and connect the uncertainty to the decision being made.

The code below translates the trial results into plain language.

In [ ]:
# Gather the key numbers
mean_diff = t_vals.mean() - c_vals.mean()

report = f"""CLINICAL TRIAL RESULTS — PLAIN LANGUAGE SUMMARY
{'=' * 52}

We compared symptom scores at 12 weeks between patients
who received the drug ({len(t_vals)} patients) and those who
received a placebo ({len(c_vals)} patients).

Patients on the drug scored an average of {abs(mean_diff):.1f} points
lower (better) than patients on the placebo.

We estimate the true difference lies between
{abs(ci_diff[1]):.1f} and {abs(ci_diff[0]):.1f} points (95% confidence interval).

This difference is statistically significant (p < 0.05),
meaning it is very unlikely to have occurred by chance alone.

In practical terms: patients on the drug showed a small but
consistent improvement in symptoms compared to the placebo.
"""

print(report)

Key principles for writing results:

- State the finding in plain language first, then give the numbers.
- Use confidence intervals to express uncertainty ("we estimate between X and Y").
- Avoid jargon where possible. If you must use a technical term, define it briefly.
- Distinguish between statistical significance and practical importance.

As the UK Government Analysis Function puts it: being upfront about uncertainty protects the integrity of your findings and ensures your audience does not draw conclusions the data cannot support.

---

## 8. Exercises

These exercises give us practice with the full confidence interval workflow: calculating intervals, comparing methods, exploring how confidence level affects width, and writing results for a real audience.

### Exercise 1: CI for score change

Calculate 95% confidence intervals for the mean **score change** (week 12 minus baseline) in each group. Print both intervals and state whether they overlap. What does the overlap (or lack of it) tell you?

In [ ]:
# Your code here


### Exercise 2: Bootstrap CI for the median difference

Use bootstrapping (10000 resamples) to build a 95% CI for the difference in **median** week 12 scores between treatment and control. Compare this to the CI for the difference in means. Are they similar?

In [ ]:
# Your code here


### Exercise 3: 90% vs 95% vs 99% confidence intervals

For the treatment group's mean week 12 score, calculate CIs at the 90%, 95%, and 99% confidence levels. Plot all three as horizontal error bars on a single chart. How does the width change as confidence increases?

In [ ]:
# Your code here


### Exercise 4: Write a non-technical summary

Using the improvement proportions from both groups (treatment and control), write a short paragraph (3-4 sentences) suitable for a hospital newsletter. Include:

- The improvement rate for each group
- The confidence interval for the treatment group's improvement rate
- A plain-language conclusion

Print your summary as a formatted string.

In [ ]:
# Your code here


---

## Summary

In this notebook we:

- Calculated **confidence intervals for means** using `scipy.stats.t.interval` and the manual formula
- Calculated **confidence intervals for proportions** using the normal approximation
- Built **bootstrap confidence intervals** by resampling with replacement and taking percentiles
- Visualised CIs with error bar plots to compare groups and sites
- Constructed a CI for the **difference in means** and checked whether it excluded zero
- Wrote results in plain language for a non-technical audience

Together with the previous two notebooks, we now have a complete toolkit for statistical inference. Notebook 1 showed us *why* inference is needed: samples vary, and that variation follows predictable patterns. Notebook 2 gave us *yes-or-no* answers about effects. This notebook gave us *how much*, expressed as a range that honestly communicates what the data supports and what it does not. That combination of rigour and honesty is what good analytical work looks like.